# 05b — Public Benchmark (HAGRID-shape)

**Audience:** Engineers and SEs who need a credibility story for the
deck: *"our numbers hold up on a human-validated public benchmark, not
just our synth-GT."*

## What this notebook shows

- A walk-through of `sentcite.benchmarks` — the adapter that transcodes
  vendor JSONL (HAGRID / ALCE) into our internal `BenchmarkItem` /
  `RetrievalResult` shape, no Azure Search roundtrip required.
- An end-to-end run against the **3-item HAGRID-shape fixture** shipped
  in the cache bundle (`data/notebook_cache/benchmarks/hagrid_fixture.jsonl`).
- The resulting `EvalReport` rendered in the **same shape as `05_evaluation`**
  so the adapter's comparability is obvious at a glance — only the data
  source differs.

## Prerequisites

- **`05_evaluation.ipynb` completed** *or* **`RESUME_FROM_CACHE = True`**
  (the default in the next cell).
- The cache bundle at `data/notebook_cache/benchmarks/hagrid_fixture.jsonl`
  (committed; loader-verified per the bundle README).

> **Live-vs-cache note.** The adapter `evaluate_benchmark()` calls
> `generate` + `cite_answer`, which both require Azure OpenAI. There is
> **no cached benchmark run artifact** in the bundle today (TODO: enrich
> the bundle with a HAGRID eval snapshot). So:
>
> - `RESUME_FROM_CACHE = True` (default) → Parts 1, 2, 4 run fully; Part 3
>   skips the live call and shows a **pre-canned expected-output block**
>   so the notebook is zero-cost on a fresh clone.
> - `RESUME_FROM_CACHE = False` → Part 3 executes the live adapter
>   against Azure. The fixture is 3 items × 2 strategies = ~6 LLM calls
>   per stage, so the cost is trivial (a few cents at most). Don't be
>   afraid to run it.

In [ ]:
# Knobs --------------------------------------------------------------
# Default: cache-resume (notebook is free to run). Flip to False to run
# the adapter live against Azure on the 3-item fixture — cost is trivial.
from pathlib import Path

RESUME_FROM_CACHE = True
LIMIT = 3  # fixture is exactly 3 items
FIXTURE_PATH = Path("data/notebook_cache/benchmarks/hagrid_fixture.jsonl")

print(f"RESUME_FROM_CACHE = {RESUME_FROM_CACHE}")
print(f"LIMIT             = {LIMIT}")
print(f"FIXTURE_PATH      = {FIXTURE_PATH}  (exists={FIXTURE_PATH.exists()})")

## Part 1 — What the benchmark adapter does

Public sentence-attribution benchmarks such as **HAGRID** (Kamalloo
et al., 2023) and **ALCE** (Gao et al., 2023) come with
human-annotated citations: for each query, a pool of candidate
passages/quotes plus a gold subset that human reviewers marked as
attribution evidence. Scoring our pipeline against those gold sets
gives us a second F1 column *independent of our synth-GT loop* — the
credibility check that our numbers aren't an artifact of
attribution-first synthetic data.

**What the adapter does** (`src/sentcite/benchmarks.py`):

1. **Loaders** (`load_hagrid_jsonl`, `load_alce_jsonl`) read the
   vendor JSONL and yield `BenchmarkItem(query, passages, gold_passage_ids)`.
2. **`build_retrieval_result(item)`** synthesizes a `RetrievalResult`
   in exactly the shape the rest of the pipeline expects — every
   passage becomes a `ChunkHit` *and* its sentences (spaCy-split)
   become `SentenceHit`s. **No Azure Search roundtrip.** The benchmark
   hands us the candidate pool already.
3. **`evaluate_benchmark(items, strategies=...)`** runs
   `generate` + `cite_answer` per item per strategy, derives
   predicted passage ids from the citation `chunk_id`s, and returns
   the same `EvalReport` object `05_evaluation` produced.

> 📎 **Real HAGRID lives at:**
> [github.com/project-miracl/hagrid](https://github.com/project-miracl/hagrid).
> The fixture in this notebook is a 3-item slice for walkthrough
> purposes only. To run the real benchmark end-to-end, see:
>
> ```bash
> python scripts/run_benchmark.py \
>     --format hagrid \
>     --input data/benchmarks/hagrid/dev.jsonl \
>     --out data/eval --run-id hagrid-dev --limit 1000
> ```

## Part 2 — Load + inspect the fixture

We render the raw JSONL side-by-side with the transcoded
`BenchmarkItem`s so the mapping — HAGRID's `quotes[].idx` →
`Passage.passage_id`, `attributable` / `attributable_quotes` →
`gold_passage_ids` — is visible end-to-end.

In [ ]:
import json
import sys
from pathlib import Path

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

import pandas as pd

fixture_path = ROOT / FIXTURE_PATH
raw_records = [json.loads(l) for l in fixture_path.read_text().splitlines() if l.strip()]

raw_df = pd.DataFrame([
    {
        "query_id": r["query_id"],
        "query": r["query"],
        "n_quotes": len(r.get("quotes", [])),
        "answer": r["answers"][0].get("answer", ""),
        "attributable_raw": (
            r["answers"][0].get("attributable")
            or sorted({
                q for s in r["answers"][0].get("sentences", [])
                for q in s.get("attributable_quotes", [])
            })
        ),
    }
    for r in raw_records
])
print(f"Raw JSONL records: {len(raw_records)}")
raw_df

In [ ]:
from sentcite.benchmarks import load_hagrid_jsonl

items = list(load_hagrid_jsonl(fixture_path, limit=LIMIT))
assert len(items) == LIMIT, f"expected {LIMIT} items, got {len(items)}"

items_df = pd.DataFrame([
    {
        "query_id": it.query_id,
        "query": it.query,
        "n_passages": len(it.passages),
        "gold_passage_ids": sorted(it.gold_passage_ids),
        "gold_answer": it.gold_answer,
        "source": it.source,
    }
    for it in items
])
print("Transcoded BenchmarkItems:")
items_df

In [ ]:
# Peek at one full item to show the Passage shape.
it = items[0]
print(f"query_id: {it.query_id}")
print(f"query:    {it.query}")
print(f"gold:     {sorted(it.gold_passage_ids)}")
print("passages:")
for p in it.passages:
    marker = "★ GOLD" if p.passage_id in it.gold_passage_ids else "      "
    print(f"  {marker} [{p.passage_id}] {p.text}")

## Part 3 — Run the adapter

The adapter's public entrypoint is
`sentcite.benchmarks.evaluate_benchmark(items, strategies=...)`. It
returns an `EvalReport` of the same type `05_evaluation` produced.

Because `evaluate_benchmark` needs Azure OpenAI (for `generate` and
`cite_answer`), the `RESUME_FROM_CACHE=True` path **does not call it**.
Instead we display a pre-canned expected-output block so this notebook
runs for free. Set `RESUME_FROM_CACHE=False` to execute live against
the 3-item fixture — the cost is trivial.

> **TODO (cache bundle):** enrich `data/notebook_cache/benchmarks/`
> with a cached `EvalReport` JSON for the fixture so this part can
> resume from cache like `05_evaluation` does. Tracked alongside
> future bundle-refresh work.

In [ ]:
report = None  # populated below

if RESUME_FROM_CACHE:
    print("RESUME_FROM_CACHE=True → skipping live adapter run.")
    print("Pre-canned expected-output shape (illustrative, not a real run):\n")
    expected_md = '''\
| Metric        | inline_prompted | post_gen_alignment |
| ---           | ---             | ---                |
| Precision     | 1.000           | 1.000              |
| Recall        | 0.833           | 0.833              |
| F1            | 0.889           | 0.889              |
| Coverage      | 1.000           | 1.000              |
| Retrieval R@k | 1.000           | 1.000              |
| Faithful %    | n/a             | n/a                |
'''
    print(expected_md)
    print("Note: headline numbers above are placeholders for shape only —")
    print("the fixture is 3 items, so a real run's numbers will be noisy.")
else:
    from sentcite.benchmarks import evaluate_benchmark

    def _progress(i, n, _buckets):
        print(f"  [{i}/{n}] done", flush=True)

    print(f"Running evaluate_benchmark on {len(items)} items (live Azure call)...")
    report = evaluate_benchmark(
        items,
        strategies=("inline_prompted", "post_gen_alignment"),
        on_progress=_progress,
    )
    print(f"\nElapsed: {report.elapsed_seconds:.1f}s")

## Part 4 — Render the report (same shape as `05_evaluation`)

The whole point of the adapter is that its output plugs into the
exact same `EvalReport` rendering code as our synth-GT eval. Swap the
data source, keep the metrics path.

In [ ]:
from IPython.display import Markdown, display

if report is None:
    display(Markdown("_No live report — re-run with `RESUME_FROM_CACHE = False` for the real table._"))
else:
    display(Markdown("### Side-by-side strategy comparison\n\n" + report.to_markdown_table()))

In [ ]:
# Strategy-level DataFrame — matches the shape used in 05_evaluation Part "Summary".
if report is None:
    print("(skipped — RESUME_FROM_CACHE=True)")
else:
    rows = []
    for strat, summ in report.strategies.items():
        rows.append({
            "strategy": strat,
            "n_items": len(summ.items),
            "precision": round(summ.macro_precision, 4),
            "recall": round(summ.macro_recall, 4),
            "f1": round(summ.macro_f1, 4),
            "coverage": round(summ.macro_coverage, 4),
            "retrieval_R@k": round(summ.macro_retrieval_recall_at_k, 4),
        })
    pd.DataFrame(rows)

> **Why the shapes match.** `evaluate_benchmark` reuses
> `ItemEval`, `StrategyEvalSummary`, and `EvalReport` verbatim from
> `sentcite.eval`. The adapter is the only difference — the metric
> code path, the summary schema, and the markdown renderer are all
> shared with `05_evaluation`. That's the credibility claim: same
> scoring, different (human-validated) gold.

## Part 5 — What to expect with real HAGRID

**Sample-size caveat.** The fixture is 3 items. A single
mis-attribution swings macro F1 by ~0.33. **Don't quote headline
numbers from this notebook to a customer.** Run the full
~1,000-item HAGRID dev slice before anything ships in a deck:

```bash
python scripts/run_benchmark.py \
    --format hagrid \
    --input data/benchmarks/hagrid/dev.jsonl \
    --out data/eval --run-id hagrid-dev
```

**Acquisition.** The vendor JSONL isn't redistributed in this repo
(license). Grab it from the HAGRID project page
(`github.com/project-miracl/hagrid`) and drop it at
`data/benchmarks/hagrid/dev.jsonl`. The loader tolerates minor shape
drift across HAGRID releases (see `_extract_hagrid_gold_quote_ids`).

**What to expect vs. synth-GT.** F1 on HAGRID is typically **lower**
than on our synth-GT — synth-GT is generated attribution-first, so
the aligner has an easier target. The gap is *the* data point:
closing it is Phase 1b's job (SME-authored GT + targeted aligner
tuning).

## What's next

- **`demo_03_metrics_that_matter.ipynb`** — the customer-facing
  story around these numbers (once it lands).
- **`docs/notebooks/`** — the index notebook that links this one
  into the broader walkthrough (soon to exist).
- **`scripts/run_benchmark.py`** — the CLI entrypoint for running a
  real HAGRID/ALCE slice and persisting an `EvalReport` to
  `data/eval/<run_id>/`.